# GSB 5544 — PA 3.2 (Extended): Distances Between Observations — every step explained
*Finding "similar" houses and colleges: each answer broken into steps, with the reasoning and the functions.*

## How to use this notebook

"Similar" has to be made precise before a computer can find it: **which variables**, **on what scale**, and **which distance formula**. Every question below is worked in numbered steps that make each of those decisions explicit, show the table before and after each transformation, and end with an interpretation. Functions are explained in *📘 Function* notes the first time they appear. The compact answer key is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).

**The one idea behind everything here.** For a target row *t* and any other row *i*, put both on the same scale and compute
- Euclidean distance: $d(i,t)=\sqrt{\sum_j (x_{ij}-x_{tj})^2}$ — straight-line distance;
- Manhattan distance: $d(i,t)=\sum_j |x_{ij}-x_{tj}|$ — sum of absolute differences.

Small distance = similar. Everything else is about choosing and preparing the $x$'s.

### How the tables are shown

Every table is printed with a caption giving its name and its full size, followed by its first rows. The **interactive page** shows the same operations with columns coloured by the table they came from (blue = left, orange = right, green = key, yellow = created) and animates each result row: [https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html](https://gato365.github.io/gsb5544_instructor_learn_prep/week_3/pandas.html). Run the next cell once; it defines the `show()` helper used throughout.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

def show(df, title, rows=8):
    """Print a caption (name and full size), then the first `rows` rows of `df`."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
    more = "" if len(df) <= rows else f"  (first {rows} shown)"
    display(Markdown(f"**{title}** — {df.shape[0]:,} rows × {df.shape[1]} columns{more}"))
    display(df.head(rows))

print("show() is ready")

show() is ready


---
## Ames — recommending similar (but cheaper) homes

**Step 0 — load and look.** The Ames file is tab-separated (`sep="\t"`). House 0 is the first row.

In [2]:
df_ames = pd.read_csv("https://dlsun.github.io/pods/data/AmesHousing.txt", sep="\t")
print(df_ames.shape)

look = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "House Style", "Neighborhood", "Year Built", "SalePrice"]
show(df_ames[look], "df_ames — selected columns (house 0 is the first row)", rows=5)

(2930, 82)


**df_ames — selected columns (house 0 is the first row)** — 2,930 rows × 8 columns  (first 5 shown)

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,House Style,Neighborhood,Year Built,SalePrice
0,1656,3,1,0,1Story,NAmes,1960,215000
1,896,2,1,0,1Story,NAmes,1961,105000
2,1329,3,1,1,1Story,NAmes,1958,172000
3,2110,3,2,1,1Story,NAmes,1968,244000
4,1629,3,2,1,2Story,Gilbert,1997,189900


---
## Ames 1 — Cheaper homes similar to house 0 on living area, bedrooms, bathrooms

**Logic.** Four decisions: (1) *which variables* — the question names them; bathrooms come in two columns, so combine them; (2) *scale* — square feet are in the thousands, bedrooms in ones, so without scaling the distance is just "difference in square feet"; (3) *distance formula* — try two and see if it matters; (4) *the price constraint* — "cheaper" is a filter applied after computing distances, not a variable inside them.

**Step 1.1 — build the bathrooms variable and look at house 0.** A half bath counts as half.

In [3]:
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 * df_ames["Half Bath"]
house0 = df_ames.loc[0]

show_cols = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "Year Built", "SalePrice"]
show(df_ames.loc[[0], show_cols], "house 0 — the target")

**house 0 — the target** — 1 rows × 7 columns

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice
0,1656,3,1.0,1Story,NAmes,1960,215000


**Step 1.2 — select the features and see why scaling is needed.** Compare the standard deviations: a "typical" difference in living area is ~500 sq ft, in bedrooms < 1.

In [4]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
X = df_ames[features].astype(float)

show(X.describe().loc[["mean", "std", "min", "max"]].round(2).reset_index().rename(columns={"index": "stat"}), "the three features on their ORIGINAL scales")

**the three features on their ORIGINAL scales** — 4 rows × 4 columns

,stat,Gr Liv Area,Bedroom AbvGr,Bathrooms
0,mean,1499.69,2.85,1.76
1,std,505.51,0.83,0.64
2,min,334.00,0.00,0.00
3,max,5642.00,8.00,5.00


**Step 1.3 — standardize.** For each column, subtract its mean and divide by its standard deviation (a **z-score**). Afterwards every column has mean 0 and SD 1, so "one unit" means "one standard deviation" in every column.

📘 **Broadcasting:** `X - X.mean()` subtracts each column's mean from every value in that column; `/ X.std()` divides likewise. No loop needed.

In [5]:
X_z = (X - X.mean()) / X.std()
show(X_z, "X_z — standardized features (mean 0, SD 1 per column)", rows=5)
print(X_z.mean().round(3).to_dict(), X_z.std().round(3).to_dict())

**X_z — standardized features (mean 0, SD 1 per column)** — 2,930 rows × 3 columns  (first 5 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms
0,0.309212,0.176064,-1.176462
1,-1.194223,-1.032058,-1.176462
2,-0.337661,0.176064,-0.398702
3,1.207317,0.176064,1.156819
4,0.255801,0.176064,1.156819


{'Gr Liv Area': 0.0, 'Bedroom AbvGr': 0.0, 'Bathrooms': -0.0} {'Gr Liv Area': 1.0, 'Bedroom AbvGr': 1.0, 'Bathrooms': 1.0}


**Step 1.4 — distance from house 0.** `X_z - X_z.loc[0]` subtracts house 0's row from *every* row (broadcasting again). Then Euclidean = square, sum across columns (`axis=1`), square-root; Manhattan = absolute value, sum across columns.

In [6]:
diff = X_z - X_z.loc[0]
df_ames["dist_euclid"]    = np.sqrt((diff ** 2).sum(axis=1))
df_ames["dist_manhattan"] = diff.abs().sum(axis=1)

show(pd.concat([X_z, diff.add_prefix("Δ "), df_ames[["dist_euclid", "dist_manhattan"]]], axis=1), "standardized values → differences from house 0 → distances", rows=5)

**standardized values → differences from house 0 → distances** — 2,930 rows × 8 columns  (first 5 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Δ Gr Liv Area,Δ Bedroom AbvGr,Δ Bathrooms,dist_euclid,dist_manhattan
0,0.309212,0.176064,-1.176462,0.000000,0.000000,0.000000,0.000000,0.000000
1,-1.194223,-1.032058,-1.176462,-1.503435,-1.208122,0.000000,1.928698,2.711557
2,-0.337661,0.176064,-0.398702,-0.646873,0.000000,0.777760,1.011610,1.424633
3,1.207317,0.176064,1.156819,0.898105,0.000000,2.333281,2.500159,3.231386
4,0.255801,0.176064,1.156819,-0.053412,0.000000,2.333281,2.333892,2.386693


**Step 1.5 — apply the price constraint, then sort.** Keep houses cheaper than house 0; the smallest distances are the recommendations.

In [7]:
cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
print(f"{len(cheaper):,} of {len(df_ames):,} houses are cheaper than house 0 (${house0['SalePrice']:,})")

show(cheaper.sort_values("dist_euclid")[show_cols + ["dist_euclid"]].round(3), "10 nearest CHEAPER houses — Euclidean, standardized", rows=10)
show(cheaper.sort_values("dist_manhattan")[show_cols + ["dist_manhattan"]].round(3), "10 nearest CHEAPER houses — Manhattan, standardized", rows=10)

2,211 of 2,930 houses are cheaper than house 0 ($215,000)


**10 nearest CHEAPER houses — Euclidean, standardized** — 2,211 rows × 8 columns  (first 10 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_euclid
1226,1661,3,1.0,SLvl,NAmes,1955,165500,0.010
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.018
1357,1666,3,1.0,2Story,OldTown,1925,161000,0.020
758,1666,3,1.0,1.5Fin,IDOTRR,1927,135000,0.020
291,1666,3,1.0,1.5Fin,SWISU,1931,100000,0.020
2637,1668,3,1.0,1.5Fin,OldTown,1948,135000,0.024
618,1644,3,1.0,1Story,NAmes,1953,167000,0.024
2700,1640,3,1.0,1Story,Sawyer,1950,131000,0.032
1529,1639,3,1.0,1.5Fin,SWISU,1936,115000,0.034
179,1633,3,1.0,1.5Fin,OldTown,1948,129000,0.045


**10 nearest CHEAPER houses — Manhattan, standardized** — 2,211 rows × 8 columns  (first 10 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_manhattan
1226,1661,3,1.0,SLvl,NAmes,1955,165500,0.010
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.018
291,1666,3,1.0,1.5Fin,SWISU,1931,100000,0.020
758,1666,3,1.0,1.5Fin,IDOTRR,1927,135000,0.020
1357,1666,3,1.0,2Story,OldTown,1925,161000,0.020
2637,1668,3,1.0,1.5Fin,OldTown,1948,135000,0.024
618,1644,3,1.0,1Story,NAmes,1953,167000,0.024
2700,1640,3,1.0,1Story,Sawyer,1950,131000,0.032
1529,1639,3,1.0,1.5Fin,SWISU,1936,115000,0.034
179,1633,3,1.0,1.5Fin,OldTown,1948,129000,0.045


**Step 1.6 — sensitivity: wrap the recipe in a function and vary the choices.** A function lets us change one decision at a time (scaling: z-score / min-max / none; metric: Euclidean / Manhattan) and compare the top-5 lists.

📘 **Min-max scaling** maps each column to 0–1: `(X - X.min()) / (X.max() - X.min())`. It is an alternative to z-scores; both remove the units problem.

In [8]:
def nearest_cheaper(features, i=0, metric="euclidean", scaling="z", k=5):
    """Indices of the k houses nearest house i on `features`, among houses cheaper than house i."""
    X = df_ames[features].astype(float)
    if scaling == "z":
        X = (X - X.mean()) / X.std()
    elif scaling == "minmax":
        X = (X - X.min()) / (X.max() - X.min())
    diff = X - X.loc[i]
    dist = np.sqrt((diff ** 2).sum(axis=1)) if metric == "euclidean" else diff.abs().sum(axis=1)
    dist = dist[df_ames["SalePrice"] < df_ames.loc[i, "SalePrice"]]
    return dist.sort_values().head(k).index.tolist()

rows = []
for scaling in ["z", "minmax", "none"]:
    for metric in ["euclidean", "manhattan"]:
        rows.append({"scaling": scaling, "metric": metric, "nearest 5 (row numbers)": nearest_cheaper(features, scaling=scaling, metric=metric)})
show(pd.DataFrame(rows), "the five nearest cheaper houses under six settings")

**the five nearest cheaper houses under six settings** — 6 rows × 3 columns

,scaling,metric,nearest 5 (row numbers)
0,z,euclidean,"[1226, 1940, 1357, 758, 291]"
1,z,manhattan,"[1226, 1940, 291, 758, 1357]"
2,minmax,euclidean,"[1226, 1940, 758, 291, 1357]"
3,minmax,manhattan,"[1226, 1940, 291, 758, 1357]"
4,none,euclidean,"[1550, 2638, 1927, 1197, 1293]"
5,none,manhattan,"[1550, 655, 1493, 1197, 1293]"


In [9]:
show(df_ames.loc[nearest_cheaper(features, scaling="none"), show_cols], "what the UNSCALED distance picks — matched on square feet only")

**what the UNSCALED distance picks — matched on square feet only** — 5 rows × 7 columns

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice
1550,1656,3,1.5,SLvl,IDOTRR,1967,126000
2638,1657,4,1.0,1.5Fin,OldTown,1920,111500
1927,1657,3,2.0,1Story,NAmes,1970,163500
1197,1656,4,2.0,1Story,NWAmes,1973,135000
1293,1656,2,2.0,1.5Fin,OldTown,1940,119164


**Answer / interpretation.** House 0 is a 1,656 sq ft, 3-bed, 1-bath 1960 ranch in North Ames sold for $215,000. With standardized features the nearest cheaper houses (rows 1226, 1940, 1357, 758, 291 …) are 1,640–1,670 sq ft, 3 bed, 1 bath — several also in North Ames — priced $100k–$165k: sensible "same house, lower price" matches.

- **Metric:** Euclidean and Manhattan return the same five houses in slightly different order → insensitive.
- **z-score vs. min-max:** same five → insensitive.
- **No scaling:** a different list, matched on square footage alone, with 2–4 bedrooms and 1–2 baths → very sensitive. Scaling is the decision that matters.
- **Sale price in the distance?** No. The goal is "like house 0 *but cheaper*". Price is the constraint we filter on; putting it in the distance would pull matches toward houses priced *like* house 0 — the opposite of a good deal.

---
## Ames 2 — Add House Style (a categorical variable)

**Logic.** You cannot subtract "1Story" from "2Story". The standard fix is **one-hot encoding**: one 0/1 column per category. Two houses with the same style then differ by 0 on all style columns; two with different styles differ by 1 in two columns, which adds √2 ≈ 1.41 to a Euclidean distance — large next to typical z-score differences, so a style mismatch is heavily penalized.

**Step 2.1 — one-hot encode.**

📘 **Function: `pd.get_dummies(series, dtype=float)`** — returns a DataFrame with one column per distinct value, 1.0 where the row has that value and 0.0 elsewhere.

In [10]:
style_dummies = pd.get_dummies(df_ames["House Style"], dtype=float)
show(pd.concat([df_ames[["House Style"]], style_dummies], axis=1), "House Style → one-hot columns", rows=5)

**House Style → one-hot columns** — 2,930 rows × 9 columns  (first 5 shown)

,House Style,1.5Fin,1.5Unf,1Story,2.5Fin,2.5Unf,2Story,SFoyer,SLvl
0,1Story,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,1Story,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,1Story,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,1Story,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,2Story,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


**Step 2.2 — glue the encoded style onto the standardized numeric features.** `pd.concat(..., axis=1)` puts the columns side by side, aligned on the row index (both frames share `df_ames`'s index, so this is safe).

In [11]:
X2 = pd.concat([X_z, style_dummies], axis=1)
show(X2, "X2 = 3 standardized numeric columns + 8 style indicators", rows=5)

**X2 = 3 standardized numeric columns + 8 style indicators** — 2,930 rows × 11 columns  (first 5 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,1.5Fin,1.5Unf,1Story,2.5Fin,2.5Unf,2Story,SFoyer,SLvl
0,0.309212,0.176064,-1.176462,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,-1.194223,-1.032058,-1.176462,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-0.337661,0.176064,-0.398702,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,1.207317,0.176064,1.156819,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.255801,0.176064,1.156819,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


**Step 2.3 — distance, filter, sort — exactly as before.**

In [12]:
diff2 = X2 - X2.loc[0]
df_ames["dist_style"] = np.sqrt((diff2 ** 2).sum(axis=1))

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
show(cheaper.sort_values("dist_style")[show_cols + ["dist_style"]].round(3), "10 nearest CHEAPER houses — with House Style", rows=10)

**10 nearest CHEAPER houses — with House Style** — 2,211 rows × 8 columns  (first 10 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_style
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.018
618,1644,3,1.0,1Story,NAmes,1953,167000,0.024
2700,1640,3,1.0,1Story,Sawyer,1950,131000,0.032
314,1687,3,1.0,1Story,Timber,1948,160000,0.061
788,1689,3,1.0,1Story,Edwards,1956,127500,0.065
2282,1622,3,1.0,1Story,Mitchel,1961,168000,0.067
2298,1608,3,1.0,1Story,Mitchel,1961,80000,0.095
1240,1570,3,1.0,1Story,NAmes,1958,166800,0.170
970,1771,3,1.0,1Story,Mitchel,1960,115000,0.227
1410,1509,3,1.0,1Story,Edwards,1956,159900,0.291


**Step 2.4 — what changed? Compare the styles of the top 10 with and without the style variable.**

In [13]:
top1 = cheaper.sort_values("dist_euclid").head(10)["House Style"].value_counts()
top2 = cheaper.sort_values("dist_style").head(10)["House Style"].value_counts()
cmp = pd.DataFrame({"part 1 (no style)": top1, "part 2 (with style)": top2}).fillna(0).astype(int).reset_index()
cmp.columns = ["House Style", "part 1 (no style)", "part 2 (with style)"]
show(cmp, "House Style of the ten nearest houses")

**House Style of the ten nearest houses** — 4 rows × 3 columns

,House Style,part 1 (no style),part 2 (with style)
0,1.5Fin,5,0
1,1Story,3,10
2,2Story,1,0
3,SLvl,1,0


**Answer / interpretation.** All ten nearest houses are now `1Story` like house 0; in part 1, seven of the ten were 1.5- or 2-story houses that merely matched on size and rooms. Because a mismatch costs √2, the style dummies behave almost like a filter: same style first, then size and rooms. If that is too strict, multiply the dummy columns by a weight below 1 (e.g. 0.5) so style becomes a preference. Metric and scaling still barely matter; **how the categorical variable is encoded** is the decision here.

---
## Ames 3 — Your own mix of quantitative and categorical variables

**Logic.** With 80 columns the temptation is to use them all; do not. Every added variable gets an equal vote after standardization, so ten near-duplicate basement columns would out-vote living area. Pick a small set that captures what a buyer reacts to — size, age, quality, land, location — with both kinds of variable.

**Chosen:** quantitative `Gr Liv Area`, `Bedroom AbvGr`, `Bathrooms`, `Year Built`, `Overall Qual`, `Lot Area`, `Garage Cars`; categorical `House Style`, `Neighborhood`, `Bldg Type`.

**Step 3.1 — encode categoricals and standardize numerics in one frame.**

📘 **`pd.get_dummies(df, columns=[...])`** — encodes only the listed columns and leaves the others as they are, so no concatenation is needed. 📘 **`fillna(0)`** — `Garage Cars` has one missing value; a `NaN` anywhere makes that house's distance `NaN`, so fill it.

In [14]:
quant = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Year Built", "Overall Qual", "Lot Area", "Garage Cars"]
categ = ["House Style", "Neighborhood", "Bldg Type"]

X3 = pd.get_dummies(df_ames[quant + categ], columns=categ, dtype=float)
X3[quant] = (X3[quant] - X3[quant].mean()) / X3[quant].std()
X3 = X3.fillna(0)
print("columns in X3:", X3.shape[1], "(7 numeric +", X3.shape[1] - 7, "indicator columns)")
show(X3.iloc[:, :12], "X3 — first 12 columns (numeric standardized, then indicators)", rows=5)

columns in X3: 48 (7 numeric + 41 indicator columns)


**X3 — first 12 columns (numeric standardized, then indicators)** — 2,930 rows × 12 columns  (first 5 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Year Built,Overall Qual,Lot Area,Garage Cars,House Style_1.5Fin,House Style_1.5Unf,House Style_1Story,House Style_2.5Fin,House Style_2.5Unf
0,0.309212,0.176064,-1.176462,-0.375473,-0.067242,2.743912,0.306594,0.0,0.0,1.0,0.0,0.0
1,-1.194223,-1.032058,-1.176462,-0.342410,-0.775946,0.187065,-1.008215,0.0,0.0,1.0,0.0,0.0
2,-0.337661,0.176064,-0.398702,-0.441599,-0.067242,0.522724,-1.008215,0.0,0.0,1.0,0.0,0.0
3,1.207317,0.176064,1.156819,-0.110970,0.641462,0.128436,0.306594,0.0,0.0,1.0,0.0,0.0
4,0.255801,0.176064,1.156819,0.847855,-0.775946,0.467268,0.306594,0.0,0.0,0.0,0.0,0.0


**Step 3.2 — distance, filter, sort; then look at house 0 on the same columns.**

In [15]:
diff3 = X3 - X3.loc[0]
df_ames["dist_full"] = np.sqrt((diff3 ** 2).sum(axis=1))

show3 = show_cols + ["Overall Qual", "Lot Area", "Garage Cars", "Bldg Type"]
show(df_ames.loc[[0], show3], "house 0 on the chosen variables")

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
show(cheaper.sort_values("dist_full")[show3 + ["dist_full"]].round(3), "10 nearest CHEAPER houses — full variable set", rows=10)

**house 0 on the chosen variables** — 1 rows × 11 columns

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,Overall Qual,Lot Area,Garage Cars,Bldg Type
0,1656,3,1.0,1Story,NAmes,1960,215000,6,31770,2.0,1Fam


**10 nearest CHEAPER houses — full variable set** — 2,211 rows × 12 columns  (first 10 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,Overall Qual,Lot Area,Garage Cars,Bldg Type,dist_full
1895,1652,3,1.5,1Story,NAmes,1959,200000,6,22002,2.0,1Fam,1.464
1013,1474,3,1.0,1Story,Gilbert,1952,115000,6,31220,2.0,1Fam,1.485
2223,1560,3,1.5,1Story,Crawfor,1960,201000,6,25485,2.0,1Fam,1.810
2294,1676,3,1.5,1Story,Mitchel,1977,196000,5,33983,2.0,1Fam,1.872
970,1771,3,1.0,1Story,Mitchel,1960,115000,5,21750,2.0,1Fam,2.042
2700,1640,3,1.0,1Story,Sawyer,1950,131000,5,21370,2.0,1Fam,2.087
639,1382,3,1.0,SLvl,NAmes,1962,176000,6,19296,2.0,1Fam,2.192
1896,1429,3,1.0,1Story,NAmes,1960,181900,6,14585,2.0,1Fam,2.227
2273,1572,3,1.5,1Story,Timber,1963,186700,5,35133,3.0,1Fam,2.248
2590,2039,3,1.5,1Story,NAmes,1941,167000,7,21299,3.0,1Fam,2.360


**Step 3.3 — sensitivity to the variable list.** House 0 sits on an unusually large lot (31,770 sq ft, 99th percentile). Drop `Lot Area` and measure how much of the top 10 survives.

In [16]:
quant_no_lot = [q for q in quant if q != "Lot Area"]
X4 = pd.get_dummies(df_ames[quant_no_lot + categ], columns=categ, dtype=float)
X4[quant_no_lot] = (X4[quant_no_lot] - X4[quant_no_lot].mean()) / X4[quant_no_lot].std()
X4 = X4.fillna(0)
df_ames["dist_no_lot"] = np.sqrt(((X4 - X4.loc[0]) ** 2).sum(axis=1))

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
a = set(cheaper.sort_values("dist_full").head(10).index)
b_ = set(cheaper.sort_values("dist_no_lot").head(10).index)
print("top-10 overlap with / without Lot Area:", len(a & b_), "of 10")
show(cheaper.sort_values("dist_no_lot")[show3 + ["dist_no_lot"]].round(3), "10 nearest CHEAPER houses — without Lot Area", rows=10)

top-10 overlap with / without Lot Area: 2 of 10


**10 nearest CHEAPER houses — without Lot Area** — 2,211 rows × 12 columns  (first 10 shown)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,Overall Qual,Lot Area,Garage Cars,Bldg Type,dist_no_lot
1240,1570,3,1.0,1Story,NAmes,1958,166800,6,13200,2.0,1Fam,0.183
618,1644,3,1.0,1Story,NAmes,1953,167000,6,9600,2.0,1Fam,0.233
1896,1429,3,1.0,1Story,NAmes,1960,181900,6,14585,2.0,1Fam,0.449
989,1414,3,1.0,1Story,NAmes,1958,176500,6,11029,2.0,1Fam,0.483
1895,1652,3,1.5,1Story,NAmes,1959,200000,6,22002,2.0,1Fam,0.779
1239,1261,3,1.0,1Story,NAmes,1958,163000,6,9120,2.0,1Fam,0.784
147,1580,3,1.5,1Story,NAmes,1959,159500,6,10032,2.0,1Fam,0.793
1216,1252,3,1.0,1Story,NAmes,1959,142000,6,10721,2.0,1Fam,0.800
1231,1537,3,1.5,1Story,NAmes,1962,174000,6,8400,2.0,1Fam,0.815
2558,1433,3,1.0,1Story,NAmes,1961,161000,5,9600,2.0,1Fam,0.835


**Answer / interpretation.** With the full set, the nearest cheaper houses are one-story single-family homes of 1,470–1,770 sq ft, 3 bedrooms, 1–1.5 baths, quality 5–6, on similarly huge lots, mostly 1950s–70s — the same *kind* of property, which is what the extra variables buy. Sensitivity is now high: removing `Lot Area` alone replaces most of the top ten, because the lot is house 0's most unusual feature and standardization makes "unusual" expensive to match. Lesson: with many variables, *which* variables you include (and their scaling) matters far more than Euclidean vs. Manhattan. Sale price stays out, for the reason in part 1.

---
## Colleges similar to Cal Poly

**Step 0 — load, index by institution, single out Cal Poly.** Setting the index to the name makes `.loc[school_name]` pull out Cal Poly's row.

In [17]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")
df_college.set_index("Institution", inplace=True)

school_name = "California Polytechnic State University-San Luis Obispo"
cp = df_college.loc[school_name]

print(df_college.shape)
show(df_college.loc[[school_name], ["City", "State", "AdmissionRate", "Undergraduates", "CarnegieClassification", "Ownership"]], "Cal Poly's row (target)")

(1957, 44)


**Cal Poly's row (target)** — 1 rows × 6 columns

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership
Institution,,,,,,
California Polytechnic State University-San Luis Obispo,San Luis Obispo,CA,0.33,21090.0,Master's Colleges & Universities: Larger Programs,Public


---
## College 1 — Admission rate and number of undergraduates

**Logic.** Two quantitative variables on wildly different scales: admission rate runs 0–1, undergraduates 5–119,000. Without scaling, a 0.4 difference in admission rate is invisible next to a difference of a few hundred students. Standardize, then Euclidean distance.

**Step 1.1 — see the scales.**

In [18]:
num = ["AdmissionRate", "Undergraduates"]
A = df_college[num]
show(A.describe().loc[["mean", "std", "min", "max"]].round(3).reset_index().rename(columns={"index": "stat"}), "two variables, two scales")

**two variables, two scales** — 4 rows × 3 columns

,stat,AdmissionRate,Undergraduates
0,mean,0.732,4155.460
1,std,0.221,7768.284
2,min,0.000,5.000
3,max,1.000,119248.000


**Step 1.2 — standardize and compute distance to Cal Poly.**

In [19]:
A_z = (A - A.mean()) / A.std()
dist1 = np.sqrt(((A_z - A_z.loc[school_name]) ** 2).sum(axis=1))

result1 = (df_college.assign(dist=dist1.round(3))
                     .sort_values("dist")[num + ["CarnegieClassification", "Ownership", "dist"]]
                     .reset_index())
show(result1, "nearest to Cal Poly — standardized Euclidean", rows=11)

**nearest to Cal Poly — standardized Euclidean** — 1,957 rows × 6 columns  (first 11 shown)

,Institution,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,dist
0,California Polytechnic State University-San Lu...,0.3300,21090.0,Master's Colleges & Universities: Larger Programs,Public,0.000
1,University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,0.309
2,DeVry University-Illinois,0.4552,19729.0,Master's Colleges & Universities: Larger Programs,Private for-profit,0.593
3,University of North Carolina at Chapel Hill,0.2040,19722.0,Doctoral Universities: Very High Research Acti...,Public,0.597
4,Clemson University,0.4922,21577.0,Doctoral Universities: Very High Research Acti...,Public,0.737
5,University of Virginia-Main Campus,0.2074,17041.0,Doctoral Universities: Very High Research Acti...,Public,0.761
6,CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761
7,Stony Brook University,0.4806,17900.0,Doctoral Universities: Very High Research Acti...,Public,0.796
8,Boston University,0.1865,17501.0,Doctoral Universities: Very High Research Acti...,Private nonprofit,0.797
9,North Carolina State University at Raleigh,0.4748,24999.0,Doctoral Universities: Very High Research Acti...,Public,0.826


**Step 1.3 — the same distance without scaling, for contrast.**

In [20]:
dist1_raw = np.sqrt(((A - A.loc[school_name]) ** 2).sum(axis=1))
show(df_college.assign(dist=dist1_raw.round(1)).sort_values("dist")[num + ["dist"]].reset_index(), "nearest to Cal Poly — UNSCALED (Undergraduates dominates)", rows=6)

**nearest to Cal Poly — UNSCALED (Undergraduates dominates)** — 1,957 rows × 4 columns  (first 6 shown)

,Institution,AdmissionRate,Undergraduates,dist
0,California Polytechnic State University-San Lu...,0.3300,21090.0,0.0
1,University of Iowa,0.8621,21198.0,108.0
2,East Carolina University,0.9389,21231.0,141.0
3,Virginia Commonwealth University,0.9277,20918.0,172.0
4,University at Buffalo,0.7009,21303.0,213.0
5,University of Kentucky,0.9401,21358.0,268.0


**Answer / interpretation.** Standardized: UC Santa Barbara, DeVry–Illinois, UNC Chapel Hill, Clemson, Virginia, CUNY Hunter, Stony Brook, Boston University — selective-ish schools with 15,000–23,000 undergraduates. Unscaled: Iowa, East Carolina, VCU, Buffalo, Kentucky — schools with ~21,000 undergraduates and *any* admission rate (many admit 70–80%). "How we decide": standardized Euclidean distance on the two variables, because that is the only version in which both variables actually count.

---
## College 2 — Add Carnegie classification and ownership

**Logic.** Same recipe as Ames 2: one-hot encode the two categoricals and append them to the standardized numerics. One wrinkle: `Institution` is **not unique** (15 duplicated names), and `pd.concat(axis=1)` aligns on the index, so it raises on duplicates. Avoid the alignment entirely by calling `pd.get_dummies` on a single frame with `columns=`.

**Step 2.1 — check the duplicate-index problem.**

In [21]:
dups = df_college.index[df_college.index.duplicated()].unique()
print("duplicated institution names:", len(dups), "→ e.g.", list(dups[:3]))

duplicated institution names: 12 → e.g. ['Emmanuel College', 'Bethel University', 'Union College']


**Step 2.2 — encode + standardize in one frame, then distance.**

In [22]:
B = pd.get_dummies(df_college[num + ["CarnegieClassification", "Ownership"]],
                   columns=["CarnegieClassification", "Ownership"], dtype=float)
B[num] = (B[num] - B[num].mean()) / B[num].std()
print("columns in B:", B.shape[1])

dist2 = np.sqrt(((B - B.loc[school_name]) ** 2).sum(axis=1))
result2 = (df_college.assign(dist=dist2.round(3))
                     .sort_values("dist")[num + ["CarnegieClassification", "Ownership", "dist"]]
                     .reset_index())
show(result2, "nearest to Cal Poly — numerics + Carnegie + ownership", rows=11)

columns in B: 35


**nearest to Cal Poly — numerics + Carnegie + ownership** — 1,957 rows × 6 columns  (first 11 shown)

,Institution,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,dist
0,California Polytechnic State University-San Lu...,0.3300,21090.0,Master's Colleges & Universities: Larger Programs,Public,0.000
1,CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761
2,CUNY Bernard M Baruch College,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.074
3,CUNY John Jay College of Criminal Justice,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.185
4,CUNY Brooklyn College,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376
5,University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.448
6,California State Polytechnic University-Pomona,0.6062,26802.0,Master's Colleges & Universities: Larger Programs,Public,1.450
7,CUNY Queens College,0.6078,14859.0,Master's Colleges & Universities: Larger Programs,Public,1.491
8,DeVry University-Illinois,0.4552,19729.0,Master's Colleges & Universities: Larger Programs,Private for-profit,1.534
9,University of North Carolina at Chapel Hill,0.2040,19722.0,Doctoral Universities: Very High Research Acti...,Public,1.535


**Answer / interpretation.** Cal Poly is a *public* "Master's Colleges & Universities: Larger Programs" school, and each of those two facts costs √2 to mismatch, so the neighbours are now CUNY Hunter, Baruch, John Jay and Brooklyn College, then Cal Poly Pomona and CUNY Queens — public master's-level institutions with 12,000–27,000 undergraduates. UCSB, UNC and UVA fall away (research doctoral universities); DeVry falls away (private for-profit). Which list is "more similar" depends on the question: *how selective and how big* → part 1; *what kind of institution* → this one.

---
## College 3 — Only the mix of fields of study (PCIP columns)

**Logic.** All 38 `PCIP` columns are proportions on the same 0–1 scale, so this time **do not standardize**: z-scoring would inflate rare fields (a 1-point difference in library science would count as much as a 20-point difference in engineering). Euclidean distance on the raw proportions; as a check, **cosine similarity**, which compares the *shape* of the mix.

**Step 3.1 — select the columns and see Cal Poly's profile.**

📘 **`df.filter(like="PCIP")`** keeps columns whose name contains the text.

In [23]:
P = df_college.filter(like="PCIP")
cp_fields = P.loc[school_name]
print("PCIP columns:", P.shape[1], "  any missing:", P.isna().any().any())

top = cp_fields.sort_values(ascending=False).head(6).reset_index()
top.columns = ["CIP field", "share of Cal Poly students"]
show(top, "Cal Poly's largest fields (14 = engineering, 52 = business, 01 = agriculture, 45 = social sciences)")

PCIP columns: 38   any missing: False


**Cal Poly's largest fields (14 = engineering, 52 = business, 01 = agriculture, 45 = social sciences)** — 6 rows × 2 columns

,CIP field,share of Cal Poly students
0,PCIP14,0.2314
1,PCIP52,0.1637
2,PCIP01,0.1084
3,PCIP45,0.0588
4,PCIP26,0.0495
5,PCIP04,0.0441


**Step 3.2 — Euclidean distance on the raw proportions.**

In [24]:
dist3 = np.sqrt(((P - cp_fields) ** 2).sum(axis=1))
result3 = (df_college.assign(dist=dist3.round(3))
                     .sort_values("dist")[["State", "Undergraduates", "CarnegieClassification", "dist"]]
                     .reset_index())
show(result3, "nearest to Cal Poly — field-of-study mix", rows=11)

**nearest to Cal Poly — field-of-study mix** — 1,957 rows × 5 columns  (first 11 shown)

,Institution,State,Undergraduates,CarnegieClassification,dist
0,California Polytechnic State University-San Lu...,CA,21090.0,Master's Colleges & Universities: Larger Programs,0.000
1,North Carolina State University at Raleigh,NC,24999.0,Doctoral Universities: Very High Research Acti...,0.084
2,Iowa State University,IA,25537.0,Doctoral Universities: Very High Research Acti...,0.085
3,University of Illinois Urbana-Champaign,IL,33889.0,Doctoral Universities: Very High Research Acti...,0.111
4,Mississippi State University,MS,18451.0,Doctoral Universities: Very High Research Acti...,0.119
5,Texas A & M University-College Station,TX,56006.0,Doctoral Universities: Very High Research Acti...,0.124
6,Clemson University,SC,21577.0,Doctoral Universities: Very High Research Acti...,0.128
7,Purdue University-Main Campus,IN,37658.0,Doctoral Universities: Very High Research Acti...,0.134
8,Virginia Polytechnic Institute and State Unive...,VA,29699.0,Doctoral Universities: Very High Research Acti...,0.138
9,Auburn University,AL,24782.0,Doctoral Universities: Very High Research Acti...,0.159


**Step 3.3 — cosine similarity as a cross-check.** Cosine similarity = (dot product of the two profiles) ÷ (product of their lengths); it is 1 when two schools have the same *proportions* of fields regardless of scale.

📘 **`P @ v`** — the `@` operator computes the dot product of every row of `P` with the vector `v` (matrix–vector multiplication).

In [25]:
norms = np.sqrt((P ** 2).sum(axis=1))
cosine = (P @ cp_fields) / (norms * norms.loc[school_name])
sim = cosine.sort_values(ascending=False).head(8).round(3).reset_index()
sim.columns = ["Institution", "cosine similarity"]
show(sim, "highest cosine similarity to Cal Poly's field mix")

**highest cosine similarity to Cal Poly's field mix** — 8 rows × 2 columns

,Institution,cosine similarity
0,California Polytechnic State University-San Lu...,1.000
1,North Carolina State University at Raleigh,0.969
2,Iowa State University,0.967
3,University of Illinois Urbana-Champaign,0.941
4,Mississippi State University,0.933
5,Texas A & M University-College Station,0.927
6,Clemson University,0.923
7,Virginia Polytechnic Institute and State Unive...,0.920


**Answer / interpretation.** Judged by *what students study*, Cal Poly's neighbours are the big public land-grant universities: NC State, Iowa State, Illinois, Mississippi State, Texas A&M, Clemson, Purdue, Virginia Tech, Auburn, West Virginia — the engineering + business + agriculture mix is rare outside that system. Cosine similarity agrees. None of these schools appeared in parts 1–2 (they are larger, doctoral, mostly less selective). The three lists are all "correct"; they answer three different questions, and choosing the variables *is* choosing the question.